Here is the complete, well-commented Python code for the Monte Carlo simulation of the Michaelis-Menten enzyme kinetics, incorporating all your specified parameters and methods.

This "instructor" version is different from the student version because it contains a few additional comments and these instructions.

In the comment lines numbered 1-6 (below), you will see important variables to run this activity.

1. These are the "known" kinetic parameters, $K_{M}$ and $V_{max}$. The current numbers were chosen at random and could be changed to values for real enzymes.
2. The substrate concentration array is used (with the parameters in #1) to generate the synthetic velocity data. Only one of these lines should be uncommented (i.e., have the `#` removed) to simulate that particular experiment. Users can also define their own scenerios.
3. The noise level is expressed as relative Gaussian noise added to the velocities. If `noise_level` is set to 0.1, then you have 10% relative noise added to your velocities. I initially wanted this value to be 0 so the students could see "perfect" data and we could discuss how real that possibility was, but the GenAI models always insisted that it couldn't be 0 for a Monte Carlo simulation. Reasonable values are 0-20% but higher values are possible and make interesting discussion points.
4. Calculate ideal velocities before noise is added.
5. Define a function for the Michaelis Menten model.
6. This sets the number of Monte Carlo trials to run. 250 trials is very fast, but is not enough to get convergence of the parameters. In other words, you will see noticable differences in the results from different runs. To keep students from getting bored and frustrated, keep this number low enough that the results come quickly but large enough to get a statistically significant result. For the purpose of this activity, pick one number (maybe 1000) and stick with that value for the entire activity. If you wanted to take the activity in a more statistical direction, you could vary the number of trials and see when the results converge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import linregress

# 1. Define true parameters
Vmax_true = 10.0
Km_true = 50.0

# 2. Define a substrate concentration array
# uncomment only one of the S_array lines to chose one of the scenarios described in the manuscript
S_array = np.array([2, 5, 10, 20, 40, 60, 80, 120, 200]) # ideal substrate sampling
# S_array = np.array([2, 5, 10, 20, 40, 60, 80, 120]) # no high [S]
# S_array = np.array([60, 70, 80, 100, 120, 150, 200]) # no low [S]
# S_array = np.array([35, 40, 45, 50, 55, 60]) # grouped around KM
# S_array = np.array([5, 20, 80]) # too few [S]


# 3. Define the noise level
# use a value of 0 for perfect data, 1e-6 for nearly perfect data
# reasonable noise levels could be 0.05-0.20 (5-20% relative Gaussian noise)
noise_level = 1e-6

# 4. Generate synthetic velocity data using the Michaelis-Menten equation
v_true = (Vmax_true * S_array) / (Km_true + S_array)

# 5. Define the Michaelis-Menten function for scipy.optimize.curve_fit
def michaelis_menten(S, Vmax, Km):
    return (Vmax * S) / (Km + S)

# 6. Monte Carlo simulation setup
# on Google Colab:
# 250 trials ~ 1 second
# 1000 trials ~ 3 seconds
# 10000 trials ~ 20 seconds
# The sweet spot for students testing multiple conditions during a class period
#   is probably about 500-5000 trials (i.e., about 5 seconds per test).
n_trials = 10000

# Dictionaries to store fitted parameters for each method
results = {
    'MM': {'Km': [], 'Vmax': []},
    'LB': {'Km': [], 'Vmax': []},
    'EH': {'Km': [], 'Vmax': []},
    'HW': {'Km': [], 'Vmax': []}
}

# Run the simulation
for i in range(n_trials):
    # 5. Add Gaussian noise to each velocity value
    noise_std = noise_level * v_true
    v_noisy = v_true + np.random.normal(loc=0.0, scale=noise_std)

    # Prevent negative or zero velocities to avoid division by zero in linearizations
    v_noisy = np.maximum(v_noisy, 1e-12)

    # a) Nonlinear Michaelis-Menten (MM) fit
    try:
        # Initial guess for Vmax and Km is provided via p0
        popt, _ = curve_fit(michaelis_menten, S_array, v_noisy, p0=[Vmax_true, Km_true])
        results['MM']['Vmax'].append(popt[0])
        results['MM']['Km'].append(popt[1])
    except RuntimeError:
        pass # Skip if curve_fit fails to converge

    # b) Lineweaver-Burk (LB) fit: 1/v = (Km/Vmax)*(1/S) + 1/Vmax
    x_lb = 1 / S_array
    y_lb = 1 / v_noisy
    slope_lb, intercept_lb, _, _, _ = linregress(x_lb, y_lb)
    vmax_lb = 1 / intercept_lb
    km_lb = slope_lb * vmax_lb
    results['LB']['Vmax'].append(vmax_lb)
    results['LB']['Km'].append(km_lb)

    # c) Eadie-Hofstee (EH) fit: v = -Km*(v/S) + Vmax
    x_eh = v_noisy / S_array
    y_eh = v_noisy
    slope_eh, intercept_eh, _, _, _ = linregress(x_eh, y_eh)
    vmax_eh = intercept_eh
    km_eh = -slope_eh
    results['EH']['Vmax'].append(vmax_eh)
    results['EH']['Km'].append(km_eh)

    # d) Hanes-Woolf (HW) fit: S/v = (1/Vmax)*S + Km/Vmax
    x_hw = S_array
    y_hw = S_array / v_noisy
    slope_hw, intercept_hw, _, _, _ = linregress(x_hw, y_hw)
    vmax_hw = 1 / slope_hw
    km_hw = intercept_hw * vmax_hw
    results['HW']['Vmax'].append(vmax_hw)
    results['HW']['Km'].append(km_hw)

# 9. Print statistical results
print("Monte Carlo Simulation Results (250 Trials)\n" + "="*60)
print(f"{'Method':<20} | {'Parameter':<10} | {'Mean':<10} | {'Std Dev':<10} | {'95% CI'}")
print("-" * 75)

methods = ['MM', 'LB', 'EH', 'HW']
for m in methods:
    for param in ['Km', 'Vmax']:
        data = np.array(results[m][param])
        mean_val = np.mean(data)
        std_val = np.std(data)

        # 95% Confidence Interval for the estimates
        lower_bound = np.percentile(data, 2.5)
        upper_bound = np.percentile(data, 97.5)

        param_label = "Km" if param == 'Km' else "Vmax"
        print(f"{m:<20} | {param_label:<10} | {mean_val:10.6f} | {std_val:10.6e} | [{lower_bound:.6f}, {upper_bound:.6f}]")
    print("-" * 75)

# 8. Plotting the native formats using the last generated 'v_noisy' dataset
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Curve Fitting for Michaelis-Menten Kinetics (One Realization)", fontsize=16)

# Extract fitted parameters from the final trial
km_fit_mm, vmax_fit_mm = results['MM']['Km'][-1], results['MM']['Vmax'][-1]
km_fit_lb, vmax_fit_lb = results['LB']['Km'][-1], results['LB']['Vmax'][-1]
km_fit_eh, vmax_fit_eh = results['EH']['Km'][-1], results['EH']['Vmax'][-1]
km_fit_hw, vmax_fit_hw = results['HW']['Km'][-1], results['HW']['Vmax'][-1]

# a) Nonlinear MM Plot (v vs S)
ax = axes[0, 0]
ax.scatter(S_array, v_noisy, color='blue', label='Noisy Data')
S_fine = np.linspace(0, max(S_array)*1.1, 100)
ax.plot(S_fine, michaelis_menten(S_fine, vmax_fit_mm, km_fit_mm), 'r-', label='MM Fit')
ax.set_title('Nonlinear Michaelis-Menten')
ax.set_xlabel('[S]')
ax.set_ylabel('v')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

# b) Lineweaver-Burk Plot (1/v vs 1/S)
ax = axes[0, 1]
ax.scatter(1/S_array, 1/v_noisy, color='green', label='Noisy Data')
x_lb_fine = np.linspace(0, max(1/S_array)*1.1, 100)
y_lb_fine = (km_fit_lb/vmax_fit_lb) * x_lb_fine + (1/vmax_fit_lb)
ax.plot(x_lb_fine, y_lb_fine, 'r-', label='LB Fit')
ax.set_title('Lineweaver-Burk Plot')
ax.set_xlabel('1/[S]')
ax.set_ylabel('1/v')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

# c) Eadie-Hofstee Plot (v vs v/S)
ax = axes[1, 0]
ax.scatter(v_noisy/S_array, v_noisy, color='orange', label='Noisy Data')
x_eh_fine = np.linspace(0, max(v_noisy/S_array)*1.1, 100)
y_eh_fine = -km_fit_eh * x_eh_fine + vmax_fit_eh
ax.plot(x_eh_fine, y_eh_fine, 'r-', label='EH Fit')
ax.set_title('Eadie-Hofstee Plot')
ax.set_xlabel('v/[S]')
ax.set_ylabel('v')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

# d) Hanes-Woolf Plot (S/v vs S)
ax = axes[1, 1]
ax.scatter(S_array, S_array/v_noisy, color='purple', label='Noisy Data')
x_hw_fine = np.linspace(0, max(S_array)*1.1, 100)
y_hw_fine = (1/vmax_fit_hw) * x_hw_fine + (km_fit_hw/vmax_fit_hw)
ax.plot(x_hw_fine, y_hw_fine, 'r-', label='HW Fit')
ax.set_title('Hanes-Woolf Plot')
ax.set_xlabel('[S]')
ax.set_ylabel('[S]/v')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

Because you defined `noise_level = 1e-6`, your velocity values remain virtually untouched. This means your parameter standard deviations will be incredibly small, and all methods will estimate values extremely close to `Km_true = 50.0` and `Vmax_true = 10.0`.

Would you like me to show you how to increase the noise level (e.g., to $0.05$ or $0.1$) to visualize how the linear transformations (like Lineweaver-Burk) drastically amplify errors compared to the nonlinear regression?